# Lab 23-07 · Simulating a Flash Translation Layer

**Predict first**, then run. The handout is [README.md](README.md).

In [1]:
import math
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
from scipy import optimize
import ssd

## Part 1 · Write amplification against over-provisioning

*Predict* the closed-form values at 7%, 14% and 28% OP, and whether greedy cleaning beats FIFO.

In [2]:
for op in (0.07, 0.14, 0.28):
    model = ssd.wa_fifo(ssd.alpha_from_op(op))
    fifo = ssd.simulate_ftl(65536, op, policy="fifo", writes_per_page=3)
    greedy = ssd.simulate_ftl(65536, op, policy="greedy", writes_per_page=3)
    print(f"OP {op:4.0%}: closed form {model:5.2f}, simulated FIFO {fifo:5.2f}, simulated greedy {greedy:5.2f}")

OP   7%: closed form  7.82, simulated FIFO  7.79, simulated greedy  6.99


OP  14%: closed form  4.25, simulated FIFO  4.24, simulated greedy  3.99


OP  28%: closed form  2.48, simulated FIFO  2.49, simulated greedy  2.40


## Part 2 · Hot and cold data

*Predict* whether 90% of writes to 10% of the LBAs helps or hurts greedy cleaning.

In [3]:
uniform = ssd.simulate_ftl(32768, 0.07, policy="greedy", writes_per_page=3)
mixed = ssd.simulate_ftl(32768, 0.07, policy="greedy", writes_per_page=3, hot_fraction=0.1)
separated = ssd.simulate_ftl(32768, 0.07, policy="greedy", writes_per_page=3, hot_fraction=0.1, separate_streams=True)
print(f"uniform {uniform:.2f}; hot/cold mixed {mixed:.2f}; hot/cold separated {separated:.2f}")

uniform 7.00; hot/cold mixed 8.08; hot/cold separated 6.44


## Part 3 · The ECC cliff

*Predict* how much tripling the correction capability raises the tolerable raw error rate.

In [4]:
p = np.logspace(-4.5, -2, 300)
plt.figure()
for t in (20, 40, 60):
    n = 8192 + 14 * t
    plt.loglog(p, [max(ssd.codeword_failure(n, t, x), 1e-30) for x in p], label=f"t = {t}")
    r = optimize.brentq(lambda x: math.log(ssd.codeword_failure(n, t, x)) + 15 * math.log(10), 1e-6, 5e-3)
    print(f"t = {t}: failure probability 1e-15 at raw error rate {r:.3e}")
plt.ylim(1e-20, 1); plt.xlabel("raw bit error rate"); plt.ylabel("codeword failure probability"); plt.legend(); plt.show()

t = 20: failure probability 1e-15 at raw error rate 2.151e-04
t = 40: failure probability 1e-15 at raw error rate 9.738e-04
t = 60: failure probability 1e-15 at raw error rate 1.977e-03
